# Metaclasses in Python (with Trading Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to **metaclasses** in Python.

You already know OOP; here we focus on **what metaclasses are, why they exist, and how to use them sensibly**, using **trading-related examples** throughout (orders, instruments, registries, validation, mini domain-specific languages).

We will build up gradually:

1. **Core ideas**: classes as objects, `type`, and basic metaclass syntax
2. **Custom metaclasses**: intercepting class creation
3. **Auto-registration** of trading strategies and order types
4. **Class-level validation and enforcement** (e.g. required attributes/methods)
5. **Declarative mini-DSLs** for trading models using metaclasses
6. **Combining metaclasses with ABCs** (and why you might or might not)
7. **Practical patterns, pitfalls, and when to avoid metaclasses**

> The goal is to make metaclasses **intuitive and practical**, not magical. Run and modify the code cells; try your own variations as you go.

If you are unsure **why metaclasses exist** or whether you “need” them, read the next section (**Why metaclasses?**) first—it ties every numbered section below to a concrete problem.

### Contents

- [Why metaclasses? Problem, necessity, roadmap](#why-metaclasses-problem-necessity-roadmap)
- [1. Core Ideas: Classes Are Objects](#1-core-ideas-classes-are-objects)
- [2. First Custom Metaclass: Logging Class Creation](#2-first-custom-metaclass-logging-class-creation)
- [3. Auto-Registering Trading Strategies with a Metaclass](#3-auto-registering-trading-strategies-with-a-metaclass)
- [4. Enforcing Class-Level Contracts with Metaclasses](#4-enforcing-class-level-contracts-with-metaclasses)
- [5. Declarative Mini-DSL with a Metaclass (Trading Models)](#5-declarative-mini-dsl-with-a-metaclass-trading-models)
- [6. Combining Metaclasses with ABCs (and When Not To)](#6-combining-metaclasses-with-abcs-and-when-not-to)
- [7. Practical Patterns, Pitfalls, and When to Avoid Metaclasses](#7-practical-patterns-pitfalls-and-when-to-avoid-metaclasses)

### Why metaclasses? Problem, necessity, roadmap

**What feels confusing** is that everyday OOP is about **instances**—you call `strategy.on_tick(...)`. Metaclasses hook a **different moment**: when the **`class` statement runs** and Python **builds the class object** (the thing you later call to construct instances).

**What problem do they solve?** Anything that should happen **once per class**, **for every subclass**, **automatically**, **before** anyone instantiates the class:

- **Registration** — e.g. “every concrete strategy class should land in a registry” without remembering to call `register(MyStrategy)` in each file.
- **Validation** — e.g. “an instrument class **must** define `tick_size` at class level”; you want a **TypeError at class definition time**, not a wrong price the first time you trade.
- **Declarative DSLs** — the class body looks like data (`max_position = IntField(...)`); the framework **collects** those descriptors, **strips** them from the namespace, and builds **`_fields`**, defaults, and validation—same *idea* as many ORMs.

Without a metaclass (or something equivalent), you usually **repeat** boilerplate, **forget** a registration line, or **discover** bad definitions **late** (first use, or a rare branch). Metaclasses let you **fail fast at import / class creation** and keep rules in **one place**.

**What is an OOP codebase “missing” without metaclasses?** **Nothing essential.** Python is complete with plain classes, inheritance, dataclasses, and decorators. Metaclasses are **optional**. You can get similar outcomes with **`__init_subclass__`**, **decorators**, **explicit registries**, or **config files**—sometimes with a bit more duplication or a weaker “cannot forget” guarantee.

**Are they really necessary?** **Rarely.** Prefer the simplest tool; use a metaclass when you need **full control** over how the class object is built (`type.__new__`), or when the **class body** is the natural **single hook** for a framework (as in §3–§5).

**How each section maps to that story**

- **§1** — **Lever**: classes are objects; **`type`** is the default class factory. Metaclasses **subclass `type`** to change construction. The **`type("Name", bases, ns)`** example is the same machinery you are about to customize.
- **§2** — **Toy**: logging when a class is created so you **see when** metaclass code runs (pedagogy, not production).
- **§3** — **Registration**: string names from config → concrete classes; the metaclass fills the map **at define time**.
- **§4** — **Class-level contracts**: ABCs excel at **instance methods**; metaclasses excel at **rules about the class dict** (required **class** attributes, invariants).
- **§5** — **Declarative models**: class statements as **configuration** compiled into metadata.
- **§6** — **ABCs + metaclass together** (`ABCMeta`) and **metaclass conflicts**.
- **§7** — **When to stop**—alternatives and team readability.

Read **§1** next with that roadmap in mind.


## 1. Core Ideas: Classes Are Objects

This section is the **intuition** behind everything above: metaclasses are not a separate language—they **replace or extend** the default metaclass, which is **`type`**. Once you see **`type(Order) is type`**, you know *where* customization plugs in.

Before touching metaclasses, you must be 100% comfortable with this idea:

> In Python, **classes are themselves objects** created by another class called a **metaclass**.

By default, the metaclass is **`type`**.

### 1.1 `type` as "class of a class"

Let's warm up with some basic introspection.

In [1]:
# 1.1 type and class objects

class Order:
    def __init__(self, symbol: str, qty: float):
        self.symbol = symbol
        self.qty = qty


order = Order("AAPL", 100)

print("Instance:", order)
print("Class:", Order)
print("type(order):", type(order))
print("type(Order):", type(Order))  # <- this is the metaclass (usually `type`)

# You can also create a class *dynamically* using type(name, bases, attrs)
DynamicOrder = type("DynamicOrder", (Order,), {"kind": "dynamic"})

print("DynamicOrder:", DynamicOrder)
print("DynamicOrder.kind:", DynamicOrder.kind)
print("type(DynamicOrder):", type(DynamicOrder))


Instance: <__main__.Order object at 0x0000015AA6F5A3C0>
Class: <class '__main__.Order'>
type(order): <class '__main__.Order'>
type(Order): <class 'type'>
DynamicOrder: <class '__main__.DynamicOrder'>
DynamicOrder.kind: dynamic
type(DynamicOrder): <class 'type'>


### 1.1 Why this example matters for metaclasses

- **`type(order)`** is the **class** of the instance (`Order`). **`type(Order)`** is the **metaclass** of the class—usually **`type`** itself.
- **`type("DynamicOrder", (Order,), {"kind": "dynamic"})`** shows that defining a class is **not magic syntax only**: it is a **call** that **builds a class object**. A **custom metaclass** is “what runs **instead of** that bare `type(...)` call” when you write `class Foo(metaclass=MyMeta)`.
- **Usefulness:** when someone asks “why not just use a function to build classes?”—you can answer: **that function is already `type`**; a metaclass **subclasses** it to **inject** registration, validation, or namespace rewriting **at that single choke point**.


### 1.2 What is a metaclass?

Very informally:

- A **class** is a *factory for instances*.
- A **metaclass** is a *factory for classes*.

In normal code you write:

```python
class Order:  # implicitly uses metaclass=type
    ...
```

Behind the scenes, Python does roughly:

```python
Order = type("Order", bases=(object,), attrs={...})
```

A **custom metaclass** is just a subclass of `type` that overrides how classes
are created or customized.

In trading systems, metaclasses can help you:

- Auto-register new order/strategy types.
- Enforce that strategy classes define certain attributes.
- Build small, declarative DSLs for things like risk models or routing rules.

**Necessity check:** none of the above **requires** a metaclass—you could use decorators or `__init_subclass__`. Metaclasses are the **lowest-level, uniform** hook on **`type.__new__`**. Use them when that hook is clearer than scattering decorators on dozens of classes.

Next we create our first custom metaclass.

## 2. First Custom Metaclass: Logging Class Creation

**Why this section exists:** before “registration” or “validation,” you need a **clear mental model of timing**. The logging metaclass answers: *exactly when* does my code run relative to `class Sub(...): ...`? (Answer: **while the class object is being constructed**, before you instantiate `Sub()`.)

**What you are *not* missing without it:** logging is only for learning. In production you would not add a metaclass just to print; you would use it once you rely on that same hook for **real** side effects (registry, checks).

We'll start with a **toy example** to see when and how a metaclass is invoked.

Steps:

1. Subclass `type` to create `LoggingMeta`.
2. Override `__new__` or `__init__` to intercept class creation.
3. Use `metaclass=LoggingMeta` in a class definition.

We'll keep it simple and just log information when a new class is defined.

In [3]:
# 2.1 Logging metaclass

class LoggingMeta(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        print(f"[LoggingMeta.__new__] Creating class {name}")
        print("  Bases:", bases)
        print("  Attributes:", list(namespace.keys()))
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        return cls


class BaseTradingModel(metaclass=LoggingMeta):
    """Base for trading-related classes using LoggingMeta."""


class OrderModel(BaseTradingModel):
    side = "BUY"
    symbol = "AAPL"

    def describe(self):
        return f"{self.side} {self.symbol}"


order_model = OrderModel()
print("Instance of OrderModel:", order_model.describe())


[LoggingMeta.__new__] Creating class BaseTradingModel
  Bases: ()
  Attributes: ['__module__', '__qualname__', '__firstlineno__', '__doc__', '__static_attributes__']
[LoggingMeta.__new__] Creating class OrderModel
  Bases: (<class '__main__.BaseTradingModel'>,)
  Attributes: ['__module__', '__qualname__', '__firstlineno__', 'side', 'symbol', 'describe', '__static_attributes__']
Instance of OrderModel: BUY AAPL


### 2.1 What you should take away from the logging example

- **`LoggingMeta.__new__` runs twice** in the output—once for `BaseTradingModel`, once for `OrderModel`—because **each `class` statement** builds a new class object. **Inheritance does not skip** the metaclass on the child.
- The **`namespace`** dict is the **class body** as captured **before** the class exists: method objects, class attributes, `__doc__`, etc. Metaclasses can **inspect or mutate** that dict **before** the final class is returned (the next sections do **useful** mutations; here we only **print**).
- **Contrast with `__init__` on instances:** instance `__init__` runs when you call `OrderModel()`; metaclass `__new__` runs when Python **defines** `OrderModel`. That distinction is the whole **reason** metaclasses exist.


### 2.2 `__new__` vs `__init__` in metaclasses

For a metaclass (subclass of `type`):

- **`__new__(mcls, name, bases, namespace, **kwargs)`**:
  - Called **before** the class object exists.
  - You receive the class name, base classes, and attribute dict.
  - You *must* return the new class object (usually via `super().__new__`).

- **`__init__(cls, name, bases, namespace, **kwargs)`**:
  - Called **after** the class object is created.
  - You can further tweak attributes on `cls`.

For most use cases, you can choose either; many people prefer `__new__` to
avoid confusion with regular class `__init__`.

Next: a practical use case for trading systems – **auto-registration**.

## 3. Auto-Registering Trading Strategies with a Metaclass

**Problem this solves:** you want **discovery by name** (`"mean_reversion"` from YAML/JSON) **without** maintaining a hand-written dict of every strategy class or putting `@register` on every file. The metaclass makes **defining the class** the **only** step—harder to forget.

**Without a metaclass:** a **decorator** (`@register_strategy`) or **`BaseStrategy.__init_subclass__`** can register subclasses too—often **preferable** for readability. A metaclass is still a good **teaching** pattern because frameworks (and this notebook) use the same hook at scale.

A very common metaclass pattern:

> When a new subclass is defined, **automatically register it** in some central registry.

This is useful for:

- Trading **strategies** that can be discovered by name.
- Order **handlers** or **routers**.
- Risk **rules** that are pluggable.

We'll build a `StrategyMeta` that:

- Keeps a **registry** mapping strategy names → classes.
- Allows you to instantiate a strategy by name at runtime.

We'll theme the example around trading strategies that react to price ticks.

In [ ]:
# 3.1 StrategyMeta: auto-registration of strategies

from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import ClassVar, Dict, Type
 

class StrategyMeta(type):
    """Metaclass that auto-registers strategies by a `name` attribute.

    Registry is stored on the metaclass itself.
    """

    # Global registry shared by all StrategyMeta classes
    _registry: Dict[str, Type["BaseStrategy"]] = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        # First, create the actual class object as normal
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # Skip registration for the abstract base class
        if not namespace.get("__abstract__", False):
            # Use explicit `name` attribute if provided, otherwise the class name
            strategy_name = getattr(cls, "name", name)
            if strategy_name in mcls._registry:
                raise ValueError(f"Duplicate strategy name: {strategy_name}")
            # Store class in registry for later lookup
            mcls._registry[strategy_name] = cls
            print(f"[StrategyMeta] Registered strategy '{strategy_name}' -> {cls}")

        return cls

    @classmethod
    def get_strategy_cls(mcls, name: str) -> Type["BaseStrategy"]:
        # Factory helper: resolve string → strategy class
        return mcls._registry[name]

    @classmethod
    def available_strategies(mcls) -> Dict[str, Type["BaseStrategy"]]:
        # Return a copy so callers don't accidentally mutate the registry
        return dict(mcls._registry)

class BaseStrategy(ABC, metaclass=StrategyMeta):
    """Base class for all trading strategies using StrategyMeta.

    Subclasses must implement `on_tick` and may override `name`.
    """

    __abstract__ = True  # prevent base from being registered

    name: ClassVar[str]

    @abstractmethod
    def on_tick(self, price: float) -> None:
        raise NotImplementedError


class BuyAndHold(BaseStrategy):
    name = "buy_and_hold"

    def __init__(self, symbol: str, quantity: float):
        self.symbol = symbol
        self.quantity = quantity
        self.entered = False

    def on_tick(self, price: float) -> None:
        # Enter the position on the first tick only
        if not self.entered:
            print(f"[BuyAndHold] BUY {self.quantity} {self.symbol} @ {price}")
            self.entered = True


class MeanReversion(BaseStrategy):
    name = "mean_reversion"

    def __init__(self, symbol: str, threshold: float):
        self.symbol = symbol
        self.threshold = threshold
        self.last_price: float | None = None

    def on_tick(self, price: float) -> None:
        # Compare current price to last seen price and trade on large moves
        if self.last_price is not None:
            diff = price - self.last_price
            if diff < -self.threshold:
                print(f"[MeanReversion] BUY {self.symbol} @ {price} (diff={diff:.2f})")
            elif diff > self.threshold:
                print(f"[MeanReversion] SELL {self.symbol} @ {price} (diff={diff:.2f})")
        self.last_price = price


print("Available strategies:", StrategyMeta.available_strategies())

# 3.2 Factory: create strategies by name at runtime

config = [
    {"type": "buy_and_hold", "symbol": "AAPL", "quantity": 100},
    {"type": "mean_reversion", "symbol": "MSFT", "threshold": 1.0},
]

strategies: list[BaseStrategy] = []

for conf in config:
    # Look up the strategy class by its string name
    cls = StrategyMeta.get_strategy_cls(conf["type"])
    # Everything except `type` is passed to the constructor
    kwargs = {k: v for k, v in conf.items() if k != "type"}
    strategies.append(cls(**kwargs))

for price in [100.0, 101.5, 99.0, 100.5]:
    print(f"\nTick: {price}")
    for strat in strategies:
        strat.on_tick(price)


### 3.2 Takeaways from `StrategyMeta`

- The metaclass **intercepts class creation** and registers each concrete subclass.
- The `BaseStrategy` is marked with `__abstract__ = True` so it's **not** registered.
- The registry lives on the **metaclass**, not on instances.
- You can now configure strategies by **string name** (e.g. from YAML/JSON).

**Why metaclasses shine here:** the **import** of a module that defines `MeanReversion` **side-effectfully** updates the registry—no separate “bootstrap” that lists all classes. **Trade-off:** implicit magic; grep-friendly teams sometimes prefer an **explicit** registry list for auditability.

This is a realistic pattern in trading infrastructure where you want to make it easy
for users to plug in new strategies without changing central code.

## 4. Enforcing Class-Level Contracts with Metaclasses

**Problem this solves:** some rules are about the **class object**, not a single instance. Example: every `Instrument` subtype must expose **`tick_size` on the class** so `price_to_ticks` works **without** an instance. An ABC can force **`price_to_ticks`** to exist, but **“must have class attr `tick_size > 0`”** is awkward there—`abstractproperty` / class-level checks are clunkier than **one metaclass pass** over the namespace.

**What you avoid:** a bad instrument class **imports successfully** but blows up **later** when someone calls `BrokenInstrument.price_to_ticks(100)`.

Metaclasses can **validate class definitions** at creation time.

Compare with ABCs:

- **ABCs** enforce that certain *methods* are implemented on **instances** (by disallowing
  instantiation of abstract classes).
- **Metaclasses** can enforce **any condition you can express about the class object**
  itself, including:
  - Required **class attributes** (e.g. `symbol`, `exchange` for an instrument).
  - Required **method signatures** or **annotations**.
  - Constraints on **inheritance** or **mixins**.

We'll build a `InstrumentMeta` that enforces some requirements on trading instrument classes.

In [ ]:
# 4.1 InstrumentMeta: enforcing required class attributes

class InstrumentMeta(type):
    """Metaclass enforcing that instruments define certain class attributes.

    Requirements:
    - `asset_class` attribute (e.g. 'equity', 'future').
    - `tick_size` attribute (float > 0).
    """

    def __new__(mcls, name, bases, namespace, **kwargs):
        # First, create the class normally
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # Only validate concrete subclasses, not the abstract base
        if not namespace.get("__abstract__", False):
            if not hasattr(cls, "asset_class"):
                raise TypeError(f"Instrument {name} must define 'asset_class'")
            if not hasattr(cls, "tick_size"):
                raise TypeError(f"Instrument {name} must define 'tick_size'")
            if not isinstance(cls.tick_size, (int, float)) or cls.tick_size <= 0:
                raise TypeError("'tick_size' must be a positive number")

        return cls


class InstrumentBase(metaclass=InstrumentMeta):
    __abstract__ = True

    @classmethod
    def price_to_ticks(cls, price: float) -> int:
        # Convert a price level into integer tick units
        return int(round(price / cls.tick_size))


class EquityInstrument(InstrumentBase):
    asset_class = "equity"
    tick_size = 0.01


class FutureInstrument(InstrumentBase):
    asset_class = "future"
    tick_size = 0.25


print("EquityInstrument.asset_class:", EquityInstrument.asset_class)
print("FutureInstrument.price_to_ticks(100.5):", FutureInstrument.price_to_ticks(100.5))


# Uncomment to see validation failure at import time:
# class BrokenInstrument(InstrumentBase):
#     pass  # missing asset_class and tick_size -> TypeError when class is defined


### 4.2 Why validate at class-creation time?

Advantages:

- **Fail fast**: errors appear when you **import** the module, not later at runtime.
- Clearer **error messages** tied to the class definition.
- You can enforce **project-wide conventions** (naming, attributes, metadata).

**Importance vs metaclasses:** `__init_subclass__` could also inspect the subclass and raise `TypeError`—so again, the metaclass is **one** way to centralize rules, not the only way. Prefer whichever your team reads more easily.

This is particularly valuable in trading codebases where incorrect configuration
(e.g. missing `tick_size`) can lead to subtle numerical bugs.

Next we'll build a slightly more advanced, **declarative mini-DSL** using a metaclass.

## 5. Declarative Mini-DSL with a Metaclass (Trading Models)

**Problem this solves:** you want authors to write **readable, table-like** models (`symbol = Field()`, `max_position = IntField(default=0)`) while instances behave like **plain objects** with real values—not with `Field` objects stuck on `self`. The metaclass **rewrites** the class: collect `Field` instances into **`_fields`**, **remove** them from the namespace, and let `__init__` use that schema.

**Without a metaclass:** **`dataclasses`** or **Pydantic** already solve 90% of this for **data** models—**use them in production** unless you are learning or need custom collection rules. This section shows **how** frameworks implement that style.

Metaclasses can turn **class definitions into declarative configuration**.

Think of popular ORMs (like Django ORM) where you write:

```python
class User(Model):
    id = IntegerField(primary_key=True)
    name = StringField()
```

and the metaclass turns this into database table metadata.

We'll build a tiny example where you define **position risk limits** declaratively,
and the metaclass collects metadata into a registry used at runtime.

In [ ]:
# 5.1 A tiny DSL for position risk models using metaclasses

from dataclasses import dataclass
from typing import Any, Dict, Type


class Field:
    """Base class for declarative fields used in models."""

    def __init__(self, default: Any | None = None):
        self.default = default  # value used when user does not pass this field


class FloatField(Field):
    pass  # here just a marker type; could add validation later


class IntField(Field):
    pass  # likewise, could enforce int types in a real system


class RiskModelMeta(type):
    """Metaclass that collects Field instances into a `_fields` dict.

    Similar in spirit to how ORMs collect model fields.
    """

    def __new__(mcls, name, bases, namespace, **kwargs):
        fields: Dict[str, Field] = {}

        # Inherit fields from base classes that also use RiskModelMeta
        for base in bases:
            if hasattr(base, "_fields"):
                fields.update(getattr(base, "_fields"))

        # Collect fields defined in this class body
        for attr_name, value in list(namespace.items()):
            if isinstance(value, Field):
                fields[attr_name] = value
                # Remove the Field object from the class dict; we'll expose
                # the actual values as normal attributes on instances instead.
                namespace.pop(attr_name)

        # Attach the collected field metadata on the class itself
        namespace["_fields"] = fields
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)
        return cls


class RiskModel(metaclass=RiskModelMeta):
    """Base class for declarative risk models."""

    _fields: Dict[str, Field]

    def __init__(self, **kwargs: Any) -> None:
        # Initialize instance attributes from field metadata + kwargs
        for name, field in self._fields.items():
            value = kwargs.get(name, field.default)
            if value is None:
                raise ValueError(f"Missing required field '{name}'")
            setattr(self, name, value)

    def as_dict(self) -> Dict[str, Any]:
        # Serialize to plain dict for logging / storage
        return {name: getattr(self, name) for name in self._fields}


class PositionRiskModel(RiskModel):
    # Declarative field definitions read by the metaclass
    symbol = Field()
    max_position = IntField(default=0)
    max_notional = FloatField(default=0.0)


model = PositionRiskModel(symbol="AAPL", max_position=1000, max_notional=100_000.0)
print("Fields:", PositionRiskModel._fields)
print("Instance dict:", model.as_dict())


### 5.2 How this mini-DSL works

- The **class body** of `PositionRiskModel` is declarative: `symbol = Field()`, etc.
- The metaclass `RiskModelMeta` **collects all `Field` instances** into `_fields`.
- The `__init__` of `RiskModel` then uses `_fields` to:
  - enforce required fields,
  - apply defaults,
  - populate instance attributes.

**Why metaclasses (here):** merging **`_fields` from base classes** and **stripping** descriptor objects from the child namespace is **cleanest** in **`__new__`** before the class object is finalized. That is the same **“compile the class body”** idea as Django models—**importance** is educational + reading third-party code, not “you must hand-roll this daily.”

This pattern underlies many **configuration-heavy** frameworks (ORMs, serializers,
config systems) and is a natural fit for trading systems with lots of metadata
(e.g. symbol configs, risk limits, routing rules).

## 6. Combining Metaclasses with ABCs (and When Not To)

**Problem this solves:** you want **both** behaviors: “**must implement `handle`**” (ABC) **and** “**auto-register each concrete handler**” (metaclass). Python’s `ABC` uses **`ABCMeta`**. If you also introduce a custom metaclass, you must **merge** them—usually by **subclassing `ABCMeta`** instead of plain `type`—so there is **one** metaclass doing both jobs.

**What you are not missing without combining:** you could register in `__init_subclass__` on an `ABC` base and avoid a custom metaclass entirely; §6 shows the **framework** style when those hooks are already metaclass-based.

You saw in the ABC notebook how `ABC`/`abstractmethod` enforce instance-level contracts.
Metaclasses operate at the **class level**.

You can combine them, but you must be aware of **metaclass conflicts** and design trade-offs.

### 6.1 Simple combination: ABC + custom metaclass

The typical pattern is:

- Create a custom metaclass that **inherits from `ABCMeta`** (which itself is a subclass of `type`).
- Use that as the metaclass for your abstract base class.

We'll create an abstract `OrderHandler` that is also **auto-registered** via a metaclass.

In [ ]:
# 6.1 ABC + custom metaclass via ABCMeta

from abc import ABCMeta, abstractmethod
from typing import Dict, Type


class OrderHandlerMeta(ABCMeta):
    """Metaclass that combines ABC behavior with auto-registration.

    Subclasses must implement `handle`, and are also discoverable by name.
    """

    _registry: Dict[str, Type["BaseOrderHandler"]] = {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        # Let ABCMeta create the class (handles abstractmethods for us)
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        # Register only non-abstract handlers
        if not namespace.get("__abstract__", False):
            handler_name = getattr(cls, "name", name)
            if handler_name in mcls._registry:
                raise ValueError(f"Duplicate handler name: {handler_name}")
            mcls._registry[handler_name] = cls
            print(f"[OrderHandlerMeta] Registered handler '{handler_name}'")

        return cls

    @classmethod
    def get_handler_cls(mcls, name: str) -> Type["BaseOrderHandler"]:
        return mcls._registry[name]


class BaseOrderHandler(metaclass=OrderHandlerMeta):
    __abstract__ = True

    @abstractmethod
    def handle(self, order: Any) -> None:
        """Process a single order (logging, routing, risk checks, etc.)."""
        raise NotImplementedError


class LoggingHandler(BaseOrderHandler):
    name = "logging"

    def handle(self, order: Any) -> None:
        print("[LoggingHandler] Handling order:", order)


class RiskCheckedHandler(BaseOrderHandler):
    name = "risk_checked"

    def __init__(self, risk_limit: float):
        self.risk_limit = risk_limit

    def handle(self, order: Any) -> None:
        print(f"[RiskCheckedHandler] Checking order against limit {self.risk_limit}")
        print("[RiskCheckedHandler] Then forwarding order:", order)


print("Handlers registry:", OrderHandlerMeta._registry)

# Look up and instantiate a handler by name
cls = OrderHandlerMeta.get_handler_cls("logging")
handler = cls()
handler.handle({"symbol": "AAPL", "qty": 100})


### 6.1 What this example demonstrates

- **`OrderHandlerMeta(ABCMeta)`** is the key line: **one** metaclass inherits **abstract** machinery from **`ABCMeta`** and adds **registration** in `__new__`.
- **`BaseOrderHandler`** never appears in the registry because **`__abstract__ = True`** skips registration—same trick as §3.
- **Importance:** you keep **static guarantees** (“cannot instantiate handler without `handle`”) **and** **runtime discovery** by string name—typical for **plugin-style** execution layers in trading systems.


### 6.2 Metaclass conflicts

Python does **not** allow a class to have two unrelated metaclasses.

If you try something like:

- base class A with metaclass `MetaA`
- base class B with metaclass `MetaB`
- subclass C(A, B)

Python will raise a **metaclass conflict** unless one metaclass is a **subclass** of the other
(or there is a common compatible metaclass).

To avoid pain:

- Prefer to centralize metaclass logic in **one metaclass** (like `OrderHandlerMeta`).
- Keep metaclasses **rare and focused**.
- Use **ABCs without custom metaclasses** where possible; only reach for metaclasses
  when you need class-creation hooks or registries.

**Why this matters for trading code:** you might mix **in-house** metaclasses with **third-party** libraries that also use metaclasses—**conflicts** then show up as obscure errors at class definition. Knowing that **`OrderHandlerMeta(ABCMeta)`** is the **approved** merge pattern saves debugging time.

## 7. Practical Patterns, Pitfalls, and When to Avoid Metaclasses

**Closing the loop on “why / necessary?”** Metaclasses are **powerful** because they run **exactly once per class** at definition time—they **centralize** registration, validation, and DSL compilation. They are **not required** for correct OOP; they are **one** implementation strategy among decorators, `__init_subclass__`, and dedicated libraries (`dataclasses`, Pydantic). **Choose metaclasses** when the **class statement** is the clearest **single hook** and simpler tools fight the design; **avoid** them when they would confuse readers more than they help.

### 7.1 Good use cases for metaclasses

- **Auto-registration** of plugins / strategies / handlers:
  - Trading strategies, order handlers, risk rules.
- **Declarative configuration / DSLs**:
  - Risk models, routing rules, symbol configs.
- **Class-level validation and invariants**:
  - Ensuring instruments define required metadata.

### 7.2 Cases where metaclasses are overkill

- Simple instance-level behavior that can be expressed with:
  - **ABCs and mixins**.
  - Plain functions or factories.
- When you just need **runtime registration** of instances (can use decorators or globals).
- When your team is not familiar with metaclasses and you can solve the problem without them.

### 7.3 Common pitfalls

- **Overusing magic**:
  - Metaclasses can make code hard to read if overused.
  - Prefer explicit, simple constructs unless there is a clear benefit.
- **Debugging complexity**:
  - Errors may appear at import time during class creation; use clear error messages.
- **Metaclass conflicts**:
  - Combining multiple frameworks with different metaclasses can be painful.

### 7.4 How to get comfortable with metaclasses

- Start with **very small experiments** like the logging example.
- Re-implement simple patterns you already know (e.g. a registry) using a metaclass.
- Compare a metaclass-based solution with an alternative approach
  (decorators, registries, ABCs) and understand trade-offs.

### 7.5 Suggested exercises

To move from theory to fluency:

- **Exercise 1**: Extend `StrategyMeta` so each strategy can declare the **symbols** it supports, and validate config.
- **Exercise 2**: Add type/shape validation to `RiskModelMeta` (e.g. `IntField` must receive ints).
- **Exercise 3**: Build a `RouteMeta` that auto-registers order routing rules like `EquityUSRoute`, `FutureEurRoute`, etc.
- **Exercise 4**: Combine ABCs and metaclasses to create an abstract `ExecutionVenue` that is also auto-registered.

When you can implement and explain these patterns comfortably, you will have a
solid, practical understanding of metaclasses suitable for real trading codebases.